<a href="https://colab.research.google.com/github/cras-lab/ML-examples/blob/main/%EB%A1%9C%EC%BB%AC%ED%8C%8C%EC%9D%BC%EB%A1%9C_DecisionTree_%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

로컬에 있는 파일을 colab에 upload하는 방법 <BR>
이 예제는 액셀을 업로드 한다.


먼저 필요한 모듈을 임포트 한다.

In [ ]:
from google.colab import files
import pandas as pd
import os

파일 팝업 다이어로그를 통해 파일을 코랩으로 업로드 한다.<BR>
예제는 하나의 액셀파일을 로드한다고 가정하고 한다.

In [ ]:
uploaded = files.upload()

Saving Loan_Train.csv to Loan_Train.csv


업로드한 파일(들) 이름 중 첫번째 이름을 filename에 저장한다.

In [ ]:
filename = list(uploaded.keys())[0]

Colab에 업로드 된 모든 내용은 /content 폴더에 저장된다.<BR>
따라서 업로드한 파일의 전체 경로를 이를 고려해 작성해야 한다.

In [ ]:
src_path = "/content/" + filename

이제 실제로 액셀파일을 읽어 들인 후 df 데이터프레임에 저장한다.<BR>
CSV는 read_csv로 .xls와 xlsx는 read_excel로 읽어야 한다.<BR>
따라서 확장자를 보고 적절한 함수를 호출한다.

In [ ]:
_, file_extension = os.path.splitext(src_path)

if file_extension == '.csv':
  df = pd.read_csv(src_path, index_col=0)
elif file_extension in ['.xls', '.xlsx']:
  df = pd.read_excel(src_path, index_col=0)
else:
  raise ValueError("Unsupported file extension: {}".format(file_extension))

제대로 읽었는지 df를 출력해 본다.

In [ ]:
print(df)

sklearn을 이용해 로컬에 있는 파일을 처리해 본다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import export_text
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

먼저, 타겟 컬럼과 데이터 컬럼을 분리한 다음 각각 X, y에 할당한다.

In [ ]:
X = df.drop(columns=['not.fully.paid'])
y = df['not.fully.paid']

데이터를 훈련과 테스트로 분할한 다음 정확도를 측정해 본다..

In [ ]:
# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

clf = DecisionTreeClassifier(random_state=50)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.737


이번에는 특징 이름이 컬럼들에 들어있으니, 그것으로 교체

In [ ]:
# feature 이름
feature_names = X.columns

IRIS 예제 모듈을 변경해, 분류를 수행해 본다.

In [ ]:
# 트리 내부 노드 정보 출력
tree = clf.tree_
for node_id in range(tree.node_count):
    if tree.feature[node_id] != -2:   # leaf node가 아니면
        feature_index = tree.feature[node_id]
        feature_name = feature_names[feature_index]
        threshold = tree.threshold[node_id]
        print(f"Node {node_id}: Feature '{feature_name}' <= {threshold:.4f}")

트리전체 구조를 텍스트 형태로 출력해 본다.

In [ ]:
# 텍스트 형태로 트리 구조 출력
tree_structure = export_text(clf, feature_names=list(feature_names))
print(tree_structure)

결정트리를 시각화 해 본다.

In [ ]:
# 결정 트리 시각화
plt.figure(figsize=(20, 12))
plot_tree(
    clf,
    feature_names=feature_names,
    class_names=['fully_paid', 'not_fully_paid'],
    filled=True
)
plt.show()